<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-07/AI_Modul_7.2_Praktikum_Logistic_Regression.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 7.2: Praktikum Regresi Logistik (Supervised Classification)

**Mata Kuliah:** AI Academy  
**Institusi:** Institut Pertanian Stiper (INSTIPER) Yogyakarta  
**Studi Kasus:** Klasifikasi Mutu Ekspor Minyak Kelapa Sawit (CPO) Berdasarkan Parameter Laboratorium Terstandar

---

### Capaian Pembelajaran Praktikum
1. Mampu mengimplementasikan fungsi aktivasi Sigmoid, fungsi rugi Binary Cross-Entropy, dan algoritma optimasi gradien regresi logistik secara manual.
2. Mampu membangun pipeline klasifikasi menggunakan pustaka Scikit-Learn lengkap dengan standarisasi fitur (`StandardScaler`).
3. Mampu mengevaluasi performa model menggunakan Matriks Konfusi, Laporan Klasifikasi, dan Kurva ROC-AUC.
4. Mampu melakukan penyetelan ambang batas keputusan (*Decision Threshold Tuning*) untuk meminimalkan kerugian finansial akibat asimetri biaya kesalahan sortasi pabrik.

## 1. Import Pustaka & Konfigurasi Grafis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, roc_auc_score, precision_recall_curve
)

# Pengaturan visualisasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 120

print("[OK] Seluruh pustaka analisis data dan machine learning berhasil dimuat!")

## 2. Pembangkitan Dataset Mutu CPO Laboratorium Pabrik Kelapa Sawit (PKS)

Fitur laboratorium:
- `FFA_Persen`: Kadar Asam Lemak Bebas / Free Fatty Acids (% bobot)
- `Kadar_Air_Persen`: Kadar air / Moisture (% bobot)
- `Kotoran_Persen`: Kadar zat pengotor padat / Impurities (% bobot)
- `Status_Lolos`: Target biner (1 = Lolos Mutu Ekspor, 0 = Ditolak/Afkir)

In [ ]:
np.random.seed(42)
n_samples = 220

# Pembangkitan variabel prediktor kontinu
ffa = np.random.uniform(1.2, 5.8, n_samples)
air = np.random.uniform(0.06, 0.42, n_samples)
kotoran = np.random.uniform(0.005, 0.065, n_samples)

# Model probabilitas laten biologis PKS
# Hubungan logit: z = 9.8 - 1.60*FFA - 10.0*Air - 55.0*Kotoran + noise
noise = np.random.normal(0, 0.50, n_samples)
z_latent = 9.8 - 1.60 * ffa - 10.0 * air - 55.0 * kotoran + noise
prob_true = 1.0 / (1.0 + np.exp(-z_latent))
status_mutu = (prob_true >= 0.50).astype(int)

df_cpo = pd.DataFrame({
    'FFA_Persen': np.round(ffa, 2),
    'Kadar_Air_Persen': np.round(air, 3),
    'Kotoran_Persen': np.round(kotoran, 4),
    'Status_Lolos': status_mutu
})

print(f"Distribusi Kelas Target:")
print(df_cpo['Status_Lolos'].value_counts())
display(df_cpo.head())

## 3. Visualisasi Eksploratif: Distribusi Parameter Berdasarkan Mutu

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
fitur_list = ['FFA_Persen', 'Kadar_Air_Persen', 'Kotoran_Persen']

for i, col in enumerate(fitur_list):
    sns.boxplot(x='Status_Lolos', y=col, data=df_cpo, hue='Status_Lolos', palette=['#d62728', '#2ca02c'], legend=False, ax=axes[i])
    axes[i].set_title(f"Distribusi {col}", fontweight='bold')
    axes[i].set_xticklabels(['Afkir (0)', 'Lolos Ekspor (1)'])

plt.tight_layout()
plt.show()

## 4. Partisi Data & Standardisasi Fitur

Karena fitur laboratorium memiliki skala yang sangat berbeda (FFA skala satuan, Kotoran skala seperseribu), standardisasi menggunakan `StandardScaler` mutlak diperlukan.

In [ ]:
X = df_cpo[fitur_list]
y = df_cpo['Status_Lolos']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Ukuran Latih  : {X_train_scaled.shape}")
print(f"Ukuran Uji    : {X_test_scaled.shape}")

## 5. Implementasi Regresi Logistik dari Nol (Algoritma Murni NumPy)

Persamaan pembaruan gradien:
$$\nabla_{\mathbf{w}} J = \frac{1}{n} \mathbf{X}^T (\hat{\mathbf{p}} - \mathbf{y})$$
$$\nabla_{b} J = \frac{1}{n} \sum_{i=1}^n (\hat{p}_i - y_i)$$

In [ ]:
class LogisticRegressionScratch:
    def __init__(self, lr: float = 0.05, n_epochs: int = 1500):
        self.lr = lr
        self.n_epochs = n_epochs
        self.w = None
        self.b = None
        self.loss_history = []

    def _sigmoid(self, z: np.ndarray) -> np.ndarray:
        return 1.0 / (1.0 + np.exp(-np.clip(z, -250, 250)))

    def fit(self, X: np.ndarray, y: np.ndarray) -> None:
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0.0
        y_arr = y.values if hasattr(y, 'values') else y

        for _ in range(self.n_epochs):
            z = X @ self.w + self.b
            p_hat = self._sigmoid(z)
            
            # Binary cross entropy loss
            loss = -np.mean(y_arr * np.log(p_hat + 1e-15) + (1 - y_arr) * np.log(1 - p_hat + 1e-15))
            self.loss_history.append(loss)

            # Gradient computation
            dw = (1.0 / n_samples) * (X.T @ (p_hat - y_arr))
            db = (1.0 / n_samples) * np.sum(p_hat - y_arr)

            # Update
            self.w -= self.lr * dw
            self.b -= self.lr * db

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        return self._sigmoid(X @ self.w + self.b)

# Pelatihan model scratch
scratch_model = LogisticRegressionScratch(lr=0.08, n_epochs=2000)
scratch_model.fit(X_train_scaled, y_train)

print("=== BOBOT HASIL REGRESI LOGISTIK NUMPY SCRATCH ===")
print(f"Bias (b)  : {scratch_model.b:.4f}")
for f, w in zip(fitur_list, scratch_model.w):
    print(f"Bobot {f:<18} : {w:+.4f}")

## 6. Pemodelan Menggunakan Scikit-Learn & Komparasi Kinerja

In [ ]:
sk_model = LogisticRegression(C=1e5, solver='lbfgs', random_state=42)
sk_model.fit(X_train_scaled, y_train)

print("=== BOBOT HASIL SCIKIT-LEARN (MLE / L-BFGS) ===")
print(f"Bias (b)  : {sk_model.intercept_[0]:.4f}")
for f, w in zip(fitur_list, sk_model.coef_[0]):
    print(f"Bobot {f:<18} : {w:+.4f}")

# Korelasi probabilitas prediksi antara NumPy Scratch dan Scikit-Learn
p_scratch = scratch_model.predict_proba(X_test_scaled)
p_sk = sk_model.predict_proba(X_test_scaled)[:, 1]
corr_p = np.corrcoef(p_scratch, p_sk)[0, 1]
print(f"\nKorelasi Probabilitas NumPy Scratch vs Sklearn: {corr_p:.5f} (Sangat Konsisten!)")

## 7. Evaluasi Klasifikasi & Kurva ROC-AUC

In [ ]:
y_pred = sk_model.predict(X_test_scaled)
cm = confusion_matrix(y_test, y_pred)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# Heatmap Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax1,
            xticklabels=['Prediksi Afkir', 'Prediksi Lolos'],
            yticklabels=['Aktual Afkir', 'Aktual Lolos'])
ax1.set_title("Matriks Konfusi (Threshold Baku tau = 0.50)", fontweight='bold')

# Kurva ROC
fpr, tpr, thresholds = roc_curve(y_test, p_sk)
auc_val = roc_auc_score(y_test, p_sk)

ax2.plot(fpr, tpr, color='#1f77b4', linewidth=2.5, label=f'Kurva ROC (AUC = {auc_val:.4f})')
ax2.plot([0, 1], [0, 1], color='red', linestyle='--', label='Garis Tebakan Acak (AUC = 0.5)')
ax2.set_title("Kurva Receiver Operating Characteristic (ROC)", fontweight='bold')
ax2.set_xlabel("False Positive Rate (FPR)")
ax2.set_ylabel("True Positive Rate (TPR / Recall)")
ax2.legend(loc='lower right')

plt.tight_layout()
plt.show()

print("=== LAPORAN METRIK KLASIFIKASI MUTU CPO ===")
print(classification_report(y_test, y_pred, target_names=['Afkir (0)', 'Lolos Ekspor (1)']))

## 8. Optimasi Ambang Batas Keputusan Berbasis Biaya Finansial PKS

Skenario biaya pabrik:
- **Biaya False Positive (FP)**: Minyak rusak diprediksi lolos ekspor. Penalti klaim internasional = **Rp 50.000.000**
- **Biaya False Negative (FN)**: Minyak bagus diprediksi afkir. Biaya re-refining tambahan = **Rp 4.000.000**

In [ ]:
c_fp = 50_000_000
c_fn = 4_000_000

thresholds_sweep = np.linspace(0.10, 0.90, 81)
total_costs = []
fp_list = []
fn_list = []

for tau in thresholds_sweep:
    y_tau = (p_sk >= tau).astype(int)
    cm_tau = confusion_matrix(y_test, y_tau)
    tn_t, fp_t, fn_t, tp_t = cm_tau.ravel()
    cost = (fp_t * c_fp) + (fn_t * c_fn)
    total_costs.append(cost)
    fp_list.append(fp_t)
    fn_list.append(fn_t)

best_idx = np.argmin(total_costs)
best_tau = thresholds_sweep[best_idx]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds_sweep, np.array(total_costs) / 1e6, color='#d62728', linewidth=2.5, label='Total Estimasi Kerugian Finansial (Juta Rp)')
ax.axvline(best_tau, color='#2ca02c', linestyle='--', linewidth=2, label=f'Ambang Batas Optimal: tau = {best_tau:.2f}')
ax.axvline(0.50, color='gray', linestyle=':', linewidth=1.5, label='Ambang Batas Default (tau = 0.50)')
ax.set_title("Penyetelan Ambang Batas Keputusan Berbasis Biaya Finansial Pabrik CPO", fontweight='bold')
ax.set_xlabel("Ambang Batas Probabilitas (Decision Threshold tau)")
ax.set_ylabel("Total Kerugian Finansial (Juta Rupiah)")
ax.legend()
plt.show()

print(f"Biaya Finansial pada Threshold Default 0.50 : Rp {total_costs[36]:,.0f}")
print(f"Biaya Finansial pada Threshold Optimal {best_tau:.2f} : Rp {total_costs[best_idx]:,.0f}")
print(f"Penghematan Efisiensi Biaya Pabrik       : Rp {total_costs[36] - total_costs[best_idx]:,.0f}")

## 9. Tugas Mandiri & Eksplorasi Mahasiswa

1. Uji sebuah sampel tangki CPO baru dengan data laboratorium: FFA = 2.4%, Kadar Air = 0.12%, dan Kotoran = 0.015%. Berapakah probabilitas minyak tersebut lolos standar ekspor?
2. Lakukan simulasi kondisi *Class Imbalance*: Modifikasi target `y` sehingga hanya 5% sampel yang lolos ekspor. Amati perubahan pada skor *Precision* dan *Recall* jika menggunakan model tanpa pembobotan versus `class_weight='balanced'`!
3. Tuliskan analisis kesimpulan Anda mengenai mengapa nilai penalti FP yang tinggi di pabrik kelapa sawit memaksa pergeseran ambang batas keputusan $\tau$ ke arah kanan ($> 0.50$).